# Lab 02 — Chapter 16: Experiment 2 — MLE vs Laplace smoothing

So sánh unigram, bigram và trigram với MLE và Laplace trên train/validation/test.

## 1. Nạp core implementation của Chapter 14

In [ ]:
from pathlib import Path
import json

project_root = Path.cwd().resolve()
while not (project_root / 'lab2').is_dir() and project_root != project_root.parent:
    project_root = project_root.parent
if not (project_root / 'lab2').is_dir():
    raise FileNotFoundError('Không tìm thấy thư mục lab2.')

core_path = project_root / 'lab2' / 'Chapter14' / 'experiments.ipynb'
core_notebook = json.loads(core_path.read_text(encoding='utf-8'))
core_source = ''.join(core_notebook['cells'][2]['source'])
exec(core_source)

DATASET_PATH = project_root / 'data' / 'c4-train.00000-of-01024-30K.json' / 'c4-train.00000-of-01024-30K.json'
DOCUMENT_LIMIT = 10_000
TRAIN_SHARE = 0.80
VALIDATION_SHARE = 0.10
MODEL_NAMES = {1: 'unigram', 2: 'bigram', 3: 'trigram'}
SMOOTHING_OPTIONS = (('mle', False), ('laplace', True))


## 2. Chia corpus thành train, validation và test

In [ ]:
def split_sentences(sentences: Sequence[Sequence[str]]) -> tuple[list[list[str]], list[list[str]], list[list[str]]]:
    """Chia tuần tự corpus thành train, validation và test theo tỉ lệ 80/10/10."""
    train_end = max(1, int(len(sentences) * TRAIN_SHARE))
    validation_end = min(len(sentences) - 1, max(train_end + 1, int(len(sentences) * (TRAIN_SHARE + VALIDATION_SHARE))))
    return [list(row) for row in sentences[:train_end]], [list(row) for row in sentences[train_end:validation_end]], [list(row) for row in sentences[validation_end:]]

document_count, sentences = load_sentences(DATASET_PATH, DOCUMENT_LIMIT)
training_set, validation_set, test_set = split_sentences(sentences)
evaluation_sets = (('train', training_set), ('validation', validation_set), ('test', test_set))
display({'documents': document_count, 'train_sentences': len(training_set), 'validation_sentences': len(validation_set), 'test_sentences': len(test_set)})


## 3. Đánh giá MLE và Laplace bằng perplexity

In [ ]:
perplexity_results = []
for order in ORDERS:
    for smoothing_name, smoothing_enabled in SMOOTHING_OPTIONS:
        model = NGramLanguageModel(order=order, smoothing=smoothing_enabled).fit(training_set)
        for split_name, evaluation_set in evaluation_sets:
            perplexity_results.append({
                'model': MODEL_NAMES[order],
                'smoothing': smoothing_name,
                'split': split_name,
                'perplexity': model.perplexity(evaluation_set),
            })
display(perplexity_results)
